In [15]:
# import opensim as osim
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d
from OA_utils.data_utils import *
import random
import pickle


Load data from file

In [ ]:
OA_data_dir  = "C:\\Users\\bakel\\Desktop\\GRFMuscleModel\\Old_Young_Walking_Data\\"
OA_filename  = 'Silder_OA_segs_normalized_filtered'
YA_data_dir  = "C:\\Users\\bakel\\Desktop\\GRFMuscleModel\\data\\"
YA_filename  = 'Ulrich_resampled_segments.pkl'
output_prefix = 'Silder_Ulrich_mixed_cop_ankle_jrf'   # prefix for saved .npz files

SKIP_SUBJECTS = {'time_resampled', 'OA4', 'OA5', 'OA23'}

# Set True to package both datasets entirely as a test set (no train/val split).
# Useful for cross-testing models trained on a different dataset.
TEST_ONLY = False

INPUT_KEYS  = ['grf_x', 'grf_y', 'grf_z', 'cop_x', 'cop_y', 'cop_z']
OUTPUT_KEYS = [ 'addbrev', 'addlong', 'addmagDist', 'addmagIsch', 'addmagMid', 
               'addmagProx', 'bflh', 'bfsh', 'edl', 'ehl', 'fdl', 'fhl', 'gaslat', 
               'gasmed', 'glmax1', 'glmax2', 'glmax3', 'glmed1', 'glmed2', 'glmed3', 
               'glmin1', 'glmin2', 'glmin3', 'grac', 'iliacus', 'perbrev', 'perlong', 
               'psoas', 'recfem', 'semimem', 'semiten', 'soleus', 'tibant', 'tibpost', 
               'vasint', 'vaslat', 'vasmed', 'achilles', 'knee_fx', 'knee_fy', 'knee_fz', 
               'ankle_fx', 'ankle_fy', 'ankle_fz'
]

# OUTPUT_KEYS = [ 'tibpost', 'tibant', 'edl', 'ehl', 'fdl', 'fhl', 'gaslat', 'gasmed',
#                 'soleus', 'perbrev', 'perlong', 'achilles', 'ankle_fx', 'ankle_fy',
#                 'ankle_fz', 'knee_fx', 'knee_fy', 'knee_fz'
# ]

ALL_KEYS = INPUT_KEYS + OUTPUT_KEYS
N_INPUTS = len(INPUT_KEYS)

with open(OA_data_dir + OA_filename, 'rb') as f:
    OA_segs = pickle.load(f)
with open(YA_data_dir + YA_filename, 'rb') as f:
    YA_segs = pickle.load(f)

RENAME_KEYS = {
    'ankle_talus_fx': 'ankle_fx',
    'ankle_talus_fy': 'ankle_fy',
    'ankle_talus_fz': 'ankle_fz',
    'knee_tibia_fx':  'knee_fx',
    'knee_tibia_fy':  'knee_fy',
    'knee_tibia_fz':  'knee_fz',
}
DROP_KEYS = {'subtalar_calc_fx', 'subtalar_calc_fy', 'subtalar_calc_fz'}

for subj in [s for s in YA_segs if s not in SKIP_SUBJECTS]:
    for old, new in RENAME_KEYS.items():
        if old in YA_segs[subj]:
            YA_segs[subj][new] = YA_segs[subj].pop(old)
    for k in DROP_KEYS:
        YA_segs[subj].pop(k, None)



In [60]:
print(YA_segs['Subject1'].keys())

dict_keys(['grf_x', 'grf_y', 'grf_z', 'cop_x', 'cop_y', 'cop_z', 'tibpost', 'tibant', 'edl', 'ehl', 'fdl', 'fhl', 'gaslat', 'gasmed', 'soleus', 'perbrev', 'perlong', 'achilles', 'ankle_fx', 'ankle_fy', 'ankle_fz', 'knee_fx', 'knee_fy', 'knee_fz'])


In [61]:
def validate_keys(segs, dataset_name, skip=SKIP_SUBJECTS):
    subjects = [s for s in segs.keys() if s not in skip]
    sample_keys = set(segs[subjects[0]].keys())
    missing  = [k for k in ALL_KEYS    if k not in sample_keys]
    uncaught = [k for k in sample_keys if k not in ALL_KEYS]
    if missing:
        print(f'{dataset_name} ERROR   — keys defined but NOT in data:    {missing}')
    else:
        print(f'{dataset_name} OK      — all INPUT_KEYS + OUTPUT_KEYS present.')
    if uncaught:
        print(f'{dataset_name} INFO    — keys in data not in ALL_KEYS:     {uncaught}')

validate_keys(OA_segs, 'OA')
validate_keys(YA_segs, 'YA')


OA OK      — all INPUT_KEYS + OUTPUT_KEYS present.
OA INFO    — keys in data not in ALL_KEYS:     ['glmax2', 'bflh', 'semimem', 'addmagIsch', 'addmagProx', 'glmax3', 'glmin1', 'semiten', 'bfsh', 'iliacus', 'psoas', 'addbrev', 'glmax1', 'vasmed', 'glmed2', 'addmagMid', 'grac', 'glmin2', 'addlong', 'vasint', 'glmin3', 'vaslat', 'glmed3', 'recfem', 'addmagDist', 'glmed1']
YA OK      — all INPUT_KEYS + OUTPUT_KEYS present.


Visualize amount of data present for each subject

In [62]:
overall_OA_segs = 0
for subj, data in OA_segs.items():
    if subj in SKIP_SUBJECTS:
        continue
    num_segments = len(data["grf_y"])
    overall_OA_segs += num_segments
    print(f"{subj}: {num_segments} segments")
print(overall_OA_segs, 'OA segments overall')

OA1: 40 segments
OA2: 11 segments
OA7: 33 segments
OA8: 40 segments
OA9: 39 segments
OA10: 30 segments
OA11: 6 segments
OA12: 18 segments
OA13: 31 segments
OA14: 32 segments
OA17: 26 segments
OA18: 35 segments
OA19: 18 segments
OA20: 9 segments
OA22: 33 segments
OA24: 16 segments
OA25: 40 segments
457 OA segments overall


In [63]:
overall_YA_segs = 0
for subj, data in YA_segs.items():
    if subj in SKIP_SUBJECTS:
        continue
    num_segments = len(data["grf_y"])
    overall_YA_segs += num_segments
    print(f"{subj}: {num_segments} segments")
print(overall_YA_segs, 'YA segments overall')

Subject1: 1357 segments
Subject2: 1517 segments
Subject3: 1526 segments
Subject4: 1338 segments
Subject5: 1482 segments
Subject6: 1544 segments
Subject7: 1539 segments
Subject8: 1581 segments
Subject9: 1450 segments
Subject10: 1558 segments
14892 YA segments overall


In [64]:
total_segs = overall_YA_segs + overall_OA_segs
print(total_segs, 'segments total')

15349 segments total


Split data by shuffling subjects

In [65]:
def count_total_segments(split_dict, key='grf_y'):
    return sum(len(split_dict[subj][key]) for subj in split_dict)

OA_subjects = [s for s in OA_segs.keys() if s not in SKIP_SUBJECTS]
YA_subjects = [s for s in YA_segs.keys() if s not in SKIP_SUBJECTS]

if TEST_ONLY:
    OA_train, OA_val = {}, {}
    YA_train, YA_val = {}, {}
    OA_test = {s: OA_segs[s] for s in OA_subjects}
    YA_test = {s: YA_segs[s] for s in YA_subjects}
    print(f'TEST_ONLY mode â€” {count_total_segments(OA_test)} OA + {count_total_segments(YA_test)} YA segments â†’ test set')

else:
    best_seed = None
    best_err = float('inf')
    best_counts = []
    best_split = None
    target_train, target_val, target_test = 0.8, 0.1, 0.1

    for seed in range(40):
        random.seed(seed)
        OA_shuf = OA_subjects.copy()
        random.shuffle(OA_shuf)
        N = len(OA_shuf)
        train_end = int(N * 0.8)
        val_end   = int(N * 0.9)
        OA_train = {s: OA_segs[s] for s in OA_shuf[:train_end]}
        OA_val   = {s: OA_segs[s] for s in OA_shuf[train_end:val_end]}
        OA_test  = {s: OA_segs[s] for s in OA_shuf[val_end:]}

        random.seed(seed)
        YA_shuf = YA_subjects.copy()
        random.shuffle(YA_shuf)
        N = len(YA_shuf)
        train_end = int(N * 0.8)
        val_end   = int(N * 0.9)
        YA_train = {s: YA_segs[s] for s in YA_shuf[:train_end]}
        YA_val   = {s: YA_segs[s] for s in YA_shuf[train_end:val_end]}
        YA_test  = {s: YA_segs[s] for s in YA_shuf[val_end:]}

        train_segs = count_total_segments(OA_train) + count_total_segments(YA_train)
        val_segs   = count_total_segments(OA_val)   + count_total_segments(YA_val)
        test_segs  = count_total_segments(OA_test)  + count_total_segments(YA_test)
        train_p = train_segs / total_segs
        val_p   = val_segs   / total_segs
        test_p  = test_segs  / total_segs
        err = abs(train_p - target_train) + abs(val_p - target_val) + abs(test_p - target_test)

        if err < best_err:
            best_err = err
            best_seed = seed
            best_counts = (train_segs, val_segs, test_segs)
            best_split = {
                "OA_train_subjs": list(OA_train.keys()),
                "OA_val_subjs":   list(OA_val.keys()),
                "OA_test_subjs":  list(OA_test.keys()),
                "YA_train_subjs": list(YA_train.keys()),
                "YA_val_subjs":   list(YA_val.keys()),
                "YA_test_subjs":  list(YA_test.keys()),
            }

        print("Best seed:", best_seed)
        print("Counts (train/val/test):", best_counts)
        print("Ratios:",
              best_counts[0]/total_segs,
              best_counts[1]/total_segs,
              best_counts[2]/total_segs)
        print("Error:", best_err)

    time_resampled = OA_segs.get("time_resampled")


Best seed: 0
Counts (train/val/test): (12146, 1637, 1566)
Ratios: 0.7913219102221644 0.1066518991465242 0.10202619063131149
Error: 0.01735617955567137
Best seed: 1
Counts (train/val/test): (12183, 1568, 1598)
Ratios: 0.7937324907160076 0.10215649227962734 0.1041110170043651
Error: 0.012535018567984876
Best seed: 1
Counts (train/val/test): (12183, 1568, 1598)
Ratios: 0.7937324907160076 0.10215649227962734 0.1041110170043651
Error: 0.012535018567984876
Best seed: 1
Counts (train/val/test): (12183, 1568, 1598)
Ratios: 0.7937324907160076 0.10215649227962734 0.1041110170043651
Error: 0.012535018567984876
Best seed: 1
Counts (train/val/test): (12183, 1568, 1598)
Ratios: 0.7937324907160076 0.10215649227962734 0.1041110170043651
Error: 0.012535018567984876
Best seed: 5
Counts (train/val/test): (12209, 1516, 1624)
Ratios: 0.7954264121441136 0.0987686494234152 0.10580493843247117
Error: 0.011609876864942428
Best seed: 5
Counts (train/val/test): (12209, 1516, 1624)
Ratios: 0.7954264121441136 0.09

In [66]:
def count_total_segments(split_dict, key='grf_y'):
    return sum(len(split_dict[subj][key]) for subj in split_dict)

if not TEST_ONLY:
    OA_train_total = count_total_segments(OA_train)
    YA_train_total = count_total_segments(YA_train)
    OA_val_total   = count_total_segments(OA_val)
    YA_val_total   = count_total_segments(YA_val)
    print("Train (OA,YA):", OA_train_total, ",", YA_train_total)
    print("Val   (OA,YA):", OA_val_total,   ",", YA_val_total)

OA_test_total = count_total_segments(OA_test)
YA_test_total = count_total_segments(YA_test)
print("Test  (OA,YA):", OA_test_total, ",", YA_test_total)


Train (OA,YA): 340 , 12072
Val   (OA,YA): 80 , 1482
Test  (OA,YA): 37 , 1338


In [67]:
def downsample_split(split_dict, target_n, seed=None, key='grf_y'):
    """
    Randomly downsample a subject dict so the total number of segments equals
    target_n.  Segments are dropped proportionally across subjects so no
    subject loses all its data.  Returns a new dict with the same structure.
    """
    rng = np.random.default_rng(seed)
    current_n = sum(len(data[key]) for data in split_dict.values())
    if target_n >= current_n:
        return split_dict

    subjects = list(split_dict.keys())
    counts = np.array([len(split_dict[s][key]) for s in subjects], dtype=float)
    allocated = np.maximum(1, np.round(counts / counts.sum() * target_n)).astype(int)

    diff = int(allocated.sum()) - target_n
    order = np.argsort(-allocated)
    for i in order:
        if diff == 0:
            break
        if diff > 0 and allocated[i] > 1:
            allocated[i] -= 1
            diff -= 1
        elif diff < 0:
            allocated[i] += 1
            diff += 1

    new_dict = {}
    for s, n_keep in zip(subjects, allocated):
        data = split_dict[s]
        n_total = len(data[key])
        idx = sorted(rng.choice(n_total, size=int(n_keep), replace=False))
        new_dict[s] = {signal: [vals[i] for i in idx] for signal, vals in data.items()}

    return new_dict


BALANCE_THRESHOLD = 0.2

def balance_splits(a_dict, b_dict, split_name, seed=None):
    na = count_total_segments(a_dict)
    nb = count_total_segments(b_dict)
    ratio = abs(na - nb) / max(na, nb)
    if ratio <= BALANCE_THRESHOLD:
        print(f"{split_name}: balanced ({na} vs {nb}, {ratio:.1%} diff) â€” no downsampling needed.")
        return a_dict, b_dict
    target = min(na, nb)
    print(f"{split_name}: imbalanced ({na} vs {nb}, {ratio:.1%} diff) â†’ downsampling larger split to {target}")
    if na > nb:
        return downsample_split(a_dict, target, seed=seed), b_dict
    else:
        return a_dict, downsample_split(b_dict, target, seed=seed)


if not TEST_ONLY:
    OA_train, YA_train = balance_splits(OA_train, YA_train, "Train", seed=best_seed)
    OA_val,   YA_val   = balance_splits(OA_val,   YA_val,   "Val",   seed=best_seed)
    OA_test,  YA_test  = balance_splits(OA_test,  YA_test,  "Test",  seed=best_seed)
    print("\nPost-balance counts:")
    print(f"  Train  OA={count_total_segments(OA_train)}  YA={count_total_segments(YA_train)}")
    print(f"  Val    OA={count_total_segments(OA_val)}   YA={count_total_segments(YA_val)}")
    print(f"  Test   OA={count_total_segments(OA_test)}   YA={count_total_segments(YA_test)}")
else:
    print("TEST_ONLY â€” skipping balance (all data retained in test set)")


Train: imbalanced (340 vs 12072, 97.2% diff) â†’ downsampling larger split to 340
Val: imbalanced (80 vs 1482, 94.6% diff) â†’ downsampling larger split to 80
Test: imbalanced (37 vs 1338, 97.2% diff) â†’ downsampling larger split to 37

Post-balance counts:
  Train  OA=340  YA=340
  Val    OA=80   YA=80
  Test   OA=37   YA=37


In [68]:
def dict_to_array(split_dict):
    packed_segments = []
    for subj, data in split_dict.items():
        num_segs = len(data[INPUT_KEYS[0]])
        for i in range(num_segs):
            sample = np.column_stack([data[k][i] for k in ALL_KEYS])
            packed_segments.append(sample)
    return np.array(packed_segments)

In [69]:
def get_subject_ranges(split_dict):
    ranges = {}
    start_idx = 0
    for subj, data in split_dict.items():
        n = len(data[INPUT_KEYS[0]])
        end_idx = start_idx + n
        ranges[subj] = (start_idx, end_idx)
        start_idx = end_idx
    return ranges

In [70]:
OA_test_ranges = get_subject_ranges(OA_test)
print(OA_test_ranges)

{'OA13': (0, 31), 'OA11': (31, 37)}


In [71]:
OA_test_arr = dict_to_array(OA_test)
YA_test_arr = dict_to_array(YA_test)
test_arr    = np.concatenate([OA_test_arr, YA_test_arr], axis=0)
print("Test:", test_arr.shape)

if not TEST_ONLY:
    OA_train_arr = dict_to_array(OA_train)
    OA_val_arr   = dict_to_array(OA_val)
    YA_train_arr = dict_to_array(YA_train)
    YA_val_arr   = dict_to_array(YA_val)
    train_arr = np.concatenate([OA_train_arr, YA_train_arr], axis=0)
    val_arr   = np.concatenate([OA_val_arr,   YA_val_arr],   axis=0)
    print("Train:", train_arr.shape)
    print("Val:",   val_arr.shape)


Test: (74, 100, 24)
Train: (680, 100, 24)
Val: (160, 100, 24)


In [72]:
X_test, y_test = test_arr[:, :, :N_INPUTS], test_arr[:, :, N_INPUTS:]
print(f"X_test shape:  {X_test.shape}")
print(f"y_test shape:  {y_test.shape}")

if not TEST_ONLY:
    X_train, y_train = train_arr[:, :, :N_INPUTS], train_arr[:, :, N_INPUTS:]
    X_val,   y_val   = val_arr[:,   :, :N_INPUTS], val_arr[:,   :, N_INPUTS:]
    print(f"X_train shape: {X_train.shape}")
    print(f"y_train shape: {y_train.shape}")
    print(f"X_val shape:   {X_val.shape}")
    print(f"y_val shape:   {y_val.shape}")


X_test shape:  (74, 100, 6)
y_test shape:  (74, 100, 18)
X_train shape: (680, 100, 6)
y_train shape: (680, 100, 18)
X_val shape:   (160, 100, 6)
y_val shape:   (160, 100, 18)


In [ ]:
np.savez(OA_data_dir + f'{output_prefix}_test_data.npz',  X_test=X_test,   y_test=y_test, output_keys = OUTPUT_KEYS)

if not TEST_ONLY:
    np.savez(OA_data_dir + f'{output_prefix}_train_data.npz', X_train=X_train, y_train=y_train)
    np.savez(OA_data_dir + f'{output_prefix}_val_data.npz',   X_val=X_val,     y_val=y_val)
